In [0]:
%sql

USE CATALOG `placementiq-campus-career-analytics`;
USE SCHEMA default;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW gold_applications AS

SELECT
    s.student_id,
    s.branch,
    s.cohort,
    s.degree,
    s.graduation_year,
    
    a.application_id,
    a.application_date,
    a.application_status,
    a.company_id,
    a.role_id,
    a.final_outcome,
    a.final_outcome_date,
    a.last_updated_timestamp,
    a.shortlist_flag,
    a.rejection_reason

FROM silver_trusted_students s

LATERAL VIEW OUTER explode(s.applications) e AS a

WHERE a.application_id IS NOT NULL
  AND TRIM(a.application_id) <> '';

In [0]:
%sql

SELECT *
FROM gold_applications

student_id,branch,cohort,degree,graduation_year,application_id,application_date,application_status,company_id,role_id,final_outcome,final_outcome_date,last_updated_timestamp,shortlist_flag,rejection_reason
S00001,EEE,2027-B,B.Tech,2027,A000001,2025-11-03,REJECTED,C013,R013-02,REJECTED,2025-11-28,2025-11-03T18:00:00Z,Y,POSITION_CLOSED
S00001,EEE,2027-B,B.Tech,2027,A001501,2026-03-04,REJECTED,C032,R032-01,NOT_SHORTLISTED,2026-03-26,2026-03-05T06:00:00Z,N,ROLE_FIT
S00001,EEE,2027-B,B.Tech,2027,A003001,2025-08-13,REJECTED,C065,R065-02,REJECTED,2025-09-18,2025-08-15T15:00:00Z,Y,MANAGERIAL_ROUND
S00001,EEE,2027-B,B.Tech,2027,A004801,2026-04-30,FINALIZED,C067,R067-04,ACCEPTED,2026-06-13,2026-05-02T17:00:00Z,Y,null
S00002,Mechanical,2027-B,B.Tech,2027,A000002,2025-11-07,REJECTED,C014,R014-04,REJECTED,2025-12-14,2025-11-09T20:00:00Z,Y,POSITION_CLOSED
S00002,Mechanical,2027-B,B.Tech,2027,A001502,2026-01-06,REJECTED,C009,R009-03,REJECTED,2026-02-05,2026-01-08T14:00:00Z,Y,TECHNICAL_ROUND
S00002,Mechanical,2027-B,B.Tech,2027,A003002,2025-10-27,REJECTED,C070,R070-01,REJECTED,2025-11-25,2025-10-27T02:00:00Z,Y,MANAGERIAL_ROUND
S00003,ECE,2027-C,B.Tech,2027,A000003,2025-09-24,REJECTED,C090,R090-03,REJECTED,2025-11-13,2025-09-25T06:00:00Z,Y,MANAGERIAL_ROUND
S00003,ECE,2027-C,B.Tech,2027,A001503,2026-04-28,REJECTED,C017,R017-03,REJECTED,2026-05-31,2026-04-28T12:00:00Z,Y,POSITION_CLOSED
S00003,ECE,2027-C,B.Tech,2027,A003003,2026-04-19,REJECTED,C001,R001-02,REJECTED,2026-05-27,2026-04-19T21:00:00Z,Y,MANAGERIAL_ROUND


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW gold_offers AS

SELECT
    s.student_id,
    s.branch,
    s.cohort,
    s.graduation_year,

    a.application_id,
    a.company_id,
    a.role_id,
    a.final_outcome,

    o.offer_id,
    o.active_flag,
    o.compensation_band_lpa,
    o.offer_date,
    o.offer_status,
    o.offer_type

FROM silver_trusted_students s

LATERAL VIEW OUTER explode(s.applications) ea AS a
LATERAL VIEW OUTER explode(a.offers) eo AS o

WHERE o.offer_id IS NOT NULL
  AND TRIM(o.offer_id) <> '';

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW gold_interviews AS

SELECT
    interview_id,
    application_id,
    student_id,
    company_id,
    role_id,
    round_number,
    round_name,
    scheduled_timestamp,
    actual_start_timestamp,
    actual_end_timestamp,
    interview_mode,
    interview_status,
    interview_result,
    interviewer_panel_code,
    update_timestamp

FROM silver_trusted_interviews

WHERE interview_id IS NOT NULL
  AND TRIM(interview_id) <> '';

In [0]:
%sql

CREATE OR REPLACE TABLE gold_total_applications
USING DELTA
AS

SELECT
    company_id,
    COUNT(DISTINCT application_id) AS total_applications,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'KPI-001' AS kpi_key,
    'Total Applications' AS kpi_name,
    'placementiq_gold_v1.0' AS _gold_schema_version

FROM gold_applications

GROUP BY company_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_total_applications
ORDER BY total_applications DESC;

company_id,total_applications,_gold_created_at,kpi_key,kpi_name,_gold_schema_version
C094,119,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0
C007,114,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0
C089,114,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0
C005,112,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0
C006,112,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0
C083,111,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0
C042,111,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0
C063,110,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0
C036,109,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0
C003,107,2026-09-18T07:56:45.402Z,KPI-001,Total Applications,placementiq_gold_v1.0


In [0]:
%sql

CREATE OR REPLACE TABLE gold_shortlisted_applications
USING DELTA
AS

SELECT
    company_id,
    COUNT(DISTINCT application_id) AS shortlisted_applications,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'KPI-002' AS kpi_key,
    'Shortlisted Applications' AS kpi_name,
    'placementiq_gold_v1.0' AS _gold_schema_version

FROM gold_applications

WHERE UPPER(TRIM(shortlist_flag))
      IN ('Y', 'YES', 'TRUE', 'SHORTLISTED')

GROUP BY company_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_shortlisted_applications
ORDER BY shortlisted_applications DESC;

company_id,shortlisted_applications,_gold_created_at,kpi_key,kpi_name,_gold_schema_version
C042,104,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0
C094,102,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0
C063,101,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0
C005,100,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0
C007,97,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0
C036,96,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0
C089,96,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0
C006,95,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0
C058,94,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0
C083,93,2026-09-18T07:57:32.847Z,KPI-002,Shortlisted Applications,placementiq_gold_v1.0


In [0]:
%sql

CREATE OR REPLACE TABLE gold_total_interviews
USING DELTA
AS

SELECT
    company_id,
    interview_mode,

    COUNT(DISTINCT interview_id) AS total_interviews,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'KPI-003' AS kpi_key,
    'Total Interviews' AS kpi_name,
    'placementiq_gold_v1.0' AS _gold_schema_version

FROM gold_interviews

WHERE interview_mode IS NOT NULL
  AND TRIM(interview_mode) <> ''

GROUP BY
    company_id,
    interview_mode;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_total_interviews
ORDER BY total_interviews DESC;

company_id,interview_mode,total_interviews,_gold_created_at,kpi_key,kpi_name,_gold_schema_version
C063,HYBRID,130,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0
C042,ONSITE,129,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0
C042,HYBRID,128,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0
C036,HYBRID,127,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0
C007,HYBRID,126,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0
C094,HYBRID,124,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0
C094,ONLINE,124,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0
C063,ONSITE,124,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0
C042,ONLINE,123,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0
C083,ONSITE,122,2026-09-18T07:58:10.658Z,KPI-003,Total Interviews,placementiq_gold_v1.0


In [0]:
%sql

CREATE OR REPLACE TABLE gold_completed_interviews
USING DELTA
AS

SELECT
    company_id,
    interview_result,

    COUNT(DISTINCT interview_id) AS completed_interviews,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'KPI-004' AS kpi_key,
    'Completed Interviews' AS kpi_name,
    'placementiq_gold_v1.0' AS _gold_schema_version

FROM gold_interviews

WHERE UPPER(TRIM(interview_status)) = 'COMPLETED'

GROUP BY
    company_id,
    interview_result;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_completed_interviews
ORDER BY completed_interviews DESC;

company_id,interview_result,completed_interviews,_gold_created_at,kpi_key,kpi_name,_gold_schema_version
C042,ADVANCED,276,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0
C094,ADVANCED,268,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0
C063,ADVANCED,264,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0
C005,ADVANCED,257,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0
C007,ADVANCED,252,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0
C083,ADVANCED,250,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0
C036,ADVANCED,250,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0
C089,ADVANCED,247,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0
C046,ADVANCED,245,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0
C003,ADVANCED,243,2026-09-18T07:58:41.007Z,KPI-004,Completed Interviews,placementiq_gold_v1.0


In [0]:
%sql

CREATE OR REPLACE TABLE gold_selected_candidates
USING DELTA
AS

SELECT
    company_id,
    branch,

    COUNT(DISTINCT application_id) AS selected_candidates,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'KPI-005' AS kpi_key,
    'Selected Candidates' AS kpi_name,
    'placementiq_gold_v1.0' AS _gold_schema_version

FROM gold_applications

WHERE UPPER(TRIM(final_outcome)) = 'SELECTED'

GROUP BY
    company_id,
    branch;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_selected_candidates
ORDER BY selected_candidates DESC;

company_id,branch,selected_candidates,_gold_created_at,kpi_key,kpi_name,_gold_schema_version


In [0]:
%sql

CREATE OR REPLACE TABLE gold_accepted_offers
USING DELTA
AS

SELECT
    company_id,
    offer_type,

    COUNT(DISTINCT offer_id) AS accepted_offers,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'KPI-006' AS kpi_key,
    'Accepted Offers' AS kpi_name,
    'placementiq_gold_v1.0' AS _gold_schema_version

FROM gold_offers

WHERE UPPER(TRIM(offer_status)) = 'ACCEPTED'

GROUP BY
    company_id,
    offer_type;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_accepted_offers
ORDER BY accepted_offers DESC;

company_id,offer_type,accepted_offers,_gold_created_at,kpi_key,kpi_name,_gold_schema_version
C046,FULL_TIME,9,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0
C042,FULL_TIME,8,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0
C093,FULL_TIME,7,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0
C005,INTERNSHIP_TO_FTE,7,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0
C006,FULL_TIME,7,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0
C012,FULL_TIME,7,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0
C079,INTERNSHIP_TO_FTE,7,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0
C096,FULL_TIME,7,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0
C098,INTERNSHIP_TO_FTE,7,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0
C090,INTERNSHIP_TO_FTE,7,2026-09-18T07:59:59.819Z,KPI-006,Accepted Offers,placementiq_gold_v1.0


In [0]:
%sql

CREATE OR REPLACE TABLE gold_joined_candidates
USING DELTA
AS

SELECT
    company_id,
    branch,

    COUNT(DISTINCT application_id) AS joined_candidates,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'KPI-007' AS kpi_key,
    'Joined Candidates' AS kpi_name,
    'placementiq_gold_v1.0' AS _gold_schema_version

FROM gold_applications

WHERE UPPER(TRIM(final_outcome)) = 'JOINED'

GROUP BY
    company_id,
    branch;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_joined_candidates
ORDER BY joined_candidates DESC;

company_id,branch,joined_candidates,_gold_created_at,kpi_key,kpi_name,_gold_schema_version
C091,ECE,3,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0
C003,ECE,3,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0
C061,EEE,3,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0
C089,IT,3,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0
C038,Mechanical,3,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0
C072,Mechanical,3,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0
C081,CSE,3,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0
C008,ECE,3,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0
C031,IT,2,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0
C002,CSE,2,2026-09-18T08:00:26.489Z,KPI-007,Joined Candidates,placementiq_gold_v1.0


In [0]:
%sql

CREATE OR REPLACE TABLE gold_average_compensation_lpa
USING DELTA
AS

SELECT
    company_id,
    offer_type,

    ROUND(
        AVG(
            CASE
                WHEN compensation_band_lpa >= 0
                 AND compensation_band_lpa <= 30
                THEN compensation_band_lpa
            END
        ),
        2
    ) AS average_compensation_lpa,

    COUNT(DISTINCT offer_id) AS valid_offer_count,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'KPI-008' AS kpi_key,
    'Average Compensation LPA' AS kpi_name,
    'placementiq_gold_v1.0' AS _gold_schema_version

FROM gold_offers

GROUP BY
    company_id,
    offer_type;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM gold_average_compensation_lpa
ORDER BY average_compensation_lpa DESC;

company_id,offer_type,average_compensation_lpa,valid_offer_count,_gold_created_at,kpi_key,kpi_name,_gold_schema_version
C056,INTERNSHIP_TO_FTE,16.6,1,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0
C082,INTERNSHIP_TO_FTE,13.8,2,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0
C038,FULL_TIME,13.18,5,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0
C057,INTERNSHIP_TO_FTE,12.18,4,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0
C100,FULL_TIME,12.15,2,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0
C004,FULL_TIME,12.15,2,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0
C023,INTERNSHIP_TO_FTE,11.7,3,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0
C095,FULL_TIME,11.7,4,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0
C092,INTERNSHIP_TO_FTE,11.59,8,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0
C065,INTERNSHIP_TO_FTE,11.51,8,2026-09-18T08:00:53.759Z,KPI-008,Average Compensation LPA,placementiq_gold_v1.0


In [0]:
%sql

SELECT 'gold_total_applications' AS table_name,
       COUNT(*) AS row_count
FROM gold_total_applications

UNION ALL

SELECT 'gold_shortlisted_applications',
       COUNT(*)
FROM gold_shortlisted_applications

UNION ALL

SELECT 'gold_total_interviews',
       COUNT(*)
FROM gold_total_interviews

UNION ALL

SELECT 'gold_completed_interviews',
       COUNT(*)
FROM gold_completed_interviews

UNION ALL

SELECT 'gold_selected_candidates',
       COUNT(*)
FROM gold_selected_candidates

UNION ALL

SELECT 'gold_accepted_offers',
       COUNT(*)
FROM gold_accepted_offers

UNION ALL

SELECT 'gold_joined_candidates',
       COUNT(*)
FROM gold_joined_candidates

UNION ALL

SELECT 'gold_average_compensation_lpa',
       COUNT(*)
FROM gold_average_compensation_lpa;

table_name,row_count
gold_total_applications,101
gold_shortlisted_applications,101
gold_total_interviews,300
gold_completed_interviews,300
gold_selected_candidates,0
gold_accepted_offers,189
gold_joined_candidates,241
gold_average_compensation_lpa,200


In [0]:
%sql
SELECT COUNT(*) FROM silver_trusted_students;

SELECT COUNT(*) FROM silver_trusted_interviews;

SELECT
    COUNT(DISTINCT company_id) AS companies
FROM gold_applications;

companies
101


In [0]:
%sql

SELECT
    (SELECT COUNT(*) FROM silver_trusted_students) AS trusted_students,
    (SELECT COUNT(*) FROM silver_trusted_interviews) AS trusted_interviews,
    (SELECT COUNT(DISTINCT company_id) FROM gold_applications) AS companies;

trusted_students,trusted_interviews,companies
1500,24989,101
